## Model Service

In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/Users/matthaei/Documents/code/python/bachelor-project


In [2]:
from src.weather_stations.weather_station_service import WeatherStationService
from src.measurements.measurement_service import MeasurementService
from src.calculation.calculation_service import CalculationService
from src.wind_turbines.wind_turbines_service import WindTurbinesService
from src.model.model_service import ModelService
from src.database.database_service import DatabaseService
from omegaconf import DictConfig, OmegaConf
from hydra import compose, initialize_config_dir
import os
import pandas as pd

/Users/matthaei/Documents/code/python/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize Hydra configuration
config_dir = os.path.abspath("./conf")

# Initialize Hydra with the config directory
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

In [4]:
database_service = DatabaseService(cfg)

In [5]:
weather_station_service = WeatherStationService(cfg, database_service)
weather_stations_df = weather_station_service.load_from_database()

2025-09-09 21:32:20.109 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:223 - Loaded 26 weather stations from database


In [6]:
measurement_service = MeasurementService(cfg, database_service, weather_stations_df)

## Generate dataset

In [7]:
model_service = ModelService(cfg, database_service, measurement_service)

In [8]:
model_service.load_dataset()

2025-09-09 21:32:37.068 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 1000000)
2025-09-09 21:32:55.264 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 2000000)
2025-09-09 21:33:16.897 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 3000000)
2025-09-09 21:33:35.552 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 4000000)
2025-09-09 21:33:53.751 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 5000000)
2025-09-09 21:34:12.214 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loa

,station_id,u,v,hour_sin,hour_cos,doy_sin,doy_cos,u_lag_1,u_lag_3,u_lag_6,...,u_roll_std_6h,v_roll_mean_3h,v_roll_std_3h,v_roll_mean_6h,v_roll_std_6h,pressure_tendency_3h,pressure_tendency_6h,temperature_tendency_3h,temperature_tendency_6h,record_date_timestamp
144,96,1.299038,0.750000,0.000000,1.000000e+00,0.034422,0.999407,0.919253,0.307818,7.070664e-01,...,0.160736,0.174752,0.510805,0.379043,0.395959,-0.6,-1.8,-1.5,-2.4,1.577923e+09
145,96,1.280250,0.225743,0.000000,1.000000e+00,0.034422,0.999407,1.039230,0.239414,8.999027e-01,...,0.166990,0.250000,0.488323,0.316667,0.383472,-0.8,-1.9,-1.0,-2.0,1.577924e+09
146,96,1.472243,0.850000,0.000000,1.000000e+00,0.034422,0.999407,1.125833,0.450000,-1.469576e-16,...,0.208206,0.608581,0.335297,0.350000,0.424716,-0.8,-1.9,-1.0,-1.9,1.577924e+09
147,96,1.221600,0.444626,0.000000,1.000000e+00,0.034422,0.999407,0.866025,0.342020,-1.592041e-16,...,0.120841,0.506790,0.316737,0.340771,0.421394,-1.0,-1.9,-0.9,-1.8,1.577925e+09
148,96,0.952628,-0.550000,0.000000,1.000000e+00,0.034422,0.999407,1.100000,0.260472,-1.836970e-16,...,0.169031,0.248209,0.720371,0.249104,0.550417,-0.5,-1.5,-0.6,-1.6,1.577926e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7474693,7389,986.700652,156.278031,-1.000000,-1.836970e-16,-0.924291,-0.381689,986.700652,986.700652,9.867007e+02,...,0.000000,156.278031,0.000000,156.278031,0.000000,0.0,0.0,-5.8,-6.2,1.757357e+09
7474694,7389,986.700652,156.278031,-1.000000,-1.836970e-16,-0.924291,-0.381689,986.700652,986.700652,9.867007e+02,...,0.000000,156.278031,0.000000,156.278031,0.000000,0.0,0.0,-6.0,-6.2,1.757357e+09
7474695,7389,986.700652,156.278031,-0.965926,2.588190e-01,-0.924291,-0.381689,986.700652,986.700652,9.867007e+02,...,0.000000,156.278031,0.000000,156.278031,0.000000,0.0,0.0,-1020.2,-1021.1,1.757358e+09
7474696,7389,986.700652,156.278031,-0.965926,2.588190e-01,-0.924291,-0.381689,986.700652,986.700652,9.867007e+02,...,0.000000,156.278031,0.000000,156.278031,0.000000,0.0,0.0,-1020.2,-1021.1,1.757359e+09


In [9]:
model_service.save_dataset_as_pickle()

In [10]:
df = pd.read_pickle("data/dataset.pkl")

In [11]:
df.columns

Index(['station_id', 'u', 'v', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos',
       'u_lag_1', 'u_lag_3', 'u_lag_6', 'u_lag_12', 'u_lag_24', 'v_lag_1',
       'v_lag_3', 'v_lag_6', 'v_lag_12', 'v_lag_24', 'u_roll_mean_3h',
       'u_roll_std_3h', 'u_roll_mean_6h', 'u_roll_std_6h', 'v_roll_mean_3h',
       'v_roll_std_3h', 'v_roll_mean_6h', 'v_roll_std_6h',
       'pressure_tendency_3h', 'pressure_tendency_6h',
       'temperature_tendency_3h', 'temperature_tendency_6h',
       'record_date_timestamp'],
      dtype='object')

In [12]:
# Create train/test split for time series data
# Use the last 20% of data chronologically as test set
df_sorted = df.sort_values(['station_id', 'record_date_timestamp'])

# Get unique timestamps and split them
unique_timestamps = df_sorted['record_date_timestamp'].unique()
unique_timestamps = sorted(unique_timestamps)

# Calculate split point (80% train, 20% test)
split_idx = int(len(unique_timestamps) * 0.8)
train_timestamps = set(unique_timestamps[:split_idx])
test_timestamps = set(unique_timestamps[split_idx:])

# Create train and test sets based on timestamps
train_df = df_sorted[df_sorted['record_date_timestamp'].isin(train_timestamps)].copy()
test_df = df_sorted[df_sorted['record_date_timestamp'].isin(test_timestamps)].copy()

print(f"Train set: {len(train_df)} rows, date range: {train_df['record_date_timestamp'].min()} to {train_df['record_date_timestamp'].max()}")
print(f"Test set: {len(test_df)} rows, date range: {test_df['record_date_timestamp'].min()} to {test_df['record_date_timestamp'].max()}")
print(f"Train stations: {train_df['station_id'].nunique()}")
print(f"Test stations: {test_df['station_id'].nunique()}")


Train set: 5976916 rows, date range: 1577923200.0 to 1721471400.0
Test set: 1494182 rows, date range: 1721472000.0 to 1757359200.0
Train stations: 25
Test stations: 25


In [13]:
from src.model.variant.lightgbm_model import LightGBMModel

model = LightGBMModel(cfg, ["u", "v"])
model.train(train_df)


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.097470
[LightGBM] [Debug] init for col-wise cost 0.000005 seconds, init for row-wise cost 0.105839 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070773 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Dense Multi-Val Bin
[LightGBM] [Info] Total Bins 6153
[LightGBM] [Info] Number of data points in the train set: 5976916, number of used features: 27
[LightGBM] [Info] Start training from score 320.252618
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 11
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug] Trained a tree

In [14]:
model.save("models/first_model")

In [15]:
model.evaluate(test_df)

/Users/matthaei/Documents/code/python/bachelor-project/src/model/variant/lightgbm_model.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  results[f"{target}_rmse"] = rmse_per_target[i]
/Users/matthaei/Documents/code/python/bachelor-project/src/model/variant/lightgbm_model.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  results[f"{target}_mae"] = mae_per_target[i]


{'rmse_overall': np.float64(1.5223450184190905),
 'mae_overall': 0.18268336722619122,
 'u_rmse': np.float64(2.0919512629579957),
 'u_mae': np.float64(0.19296884509700155),
 'v_rmse': np.float64(0.5087323693449126),
 'v_mae': np.float64(0.17239788935538086)}

In [16]:
model.evaluate(test_df)

/Users/matthaei/Documents/code/python/bachelor-project/src/model/variant/lightgbm_model.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  results[f"{target}_rmse"] = rmse_per_target[i]
/Users/matthaei/Documents/code/python/bachelor-project/src/model/variant/lightgbm_model.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  results[f"{target}_mae"] = mae_per_target[i]


{'rmse_overall': np.float64(1.5223450184190905),
 'mae_overall': 0.18268336722619122,
 'u_rmse': np.float64(2.0919512629579957),
 'u_mae': np.float64(0.19296884509700155),
 'v_rmse': np.float64(0.5087323693449126),
 'v_mae': np.float64(0.17239788935538086)}